# pynq_puf3 实板测试

本 Notebook 调用 `src/pynq_puf3/board_test.py`。
它会执行 Overlay 身份检查、SW/HW/ISE KAT、mcycle benchmark 和 PUF enroll/reconstruct/HWKEY 流程，
并把完整证据保存到 `data/pynq_puf3/notebook_<时间>/`。

请在 PYNQ 板上确认 candidate 目录和仓库路径可访问，然后按顺序执行全部单元。

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib.util
import json
import os
import platform
import sys

REPO = Path(os.environ.get("ASCON_REPO", "/home/xilinx/jupyter_notebooks/amd_fpga_2026")).resolve()
CANDIDATE = REPO / "board/vivado_zynq7000_xc7z020_pynq_puf3/candidate"
FULL_KAT = False
WARMUP = 5
REPETITIONS = 30
KAT_COUNT = 529
RECONSTRUCTS = 5

assert (REPO / "src/pynq_puf3/board_test.py").is_file(), (
    f"仓库路径不正确：{REPO}; 请设置 ASCON_REPO"
)
assert (CANDIDATE / "pynq_puf3.bit").is_file(), f"找不到 candidate bit：{CANDIDATE}"
assert (CANDIDATE / "pynq_puf3.hwh").is_file(), f"找不到 candidate hwh：{CANDIDATE}"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = REPO / "data/pynq_puf3" / f"notebook_{run_id}"
OUT.mkdir(parents=True, exist_ok=False)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("repository:", REPO)
print("candidate:", CANDIDATE)
print("output:", OUT)
print("python:", platform.python_version())

## 执行完整上板测试

默认使用 6 个代表性 KAT。将 `FULL_KAT = True` 后重新执行配置单元，可改为运行全部 289 个容量内 KAT。
测试失败时也会先保存 `result.json` 和 `mailbox_log.txt`，便于定位问题。

In [ ]:
test_path = REPO / "src/pynq_puf3/board_test.py"
spec = importlib.util.spec_from_file_location("pynq_puf3_board_test", test_path)
board_test = importlib.util.module_from_spec(spec)
spec.loader.exec_module(board_test)

sys.argv = [
    str(test_path),
    "--candidate", str(CANDIDATE),
    "--out", str(OUT),
]
if FULL_KAT:
    sys.argv.append("--full-kat")
sys.argv.extend([
    "--warmup", str(WARMUP),
    "--repetitions", str(REPETITIONS),
    "--kat", str(KAT_COUNT),
    "--reconstructs", str(RECONSTRUCTS),
])

exit_code = board_test.main()
result = json.loads((OUT / "result.json").read_text(encoding="utf-8"))
print(json.dumps({
    "status": result.get("status"),
    "exit_code": exit_code,
    "output": str(OUT),
    "error": result.get("error"),
}, indent=2, ensure_ascii=False))
if exit_code != 0:
    raise RuntimeError(f"pynq_puf3 board test failed; evidence: {OUT}")

In [ ]:
print("status:", result["status"])
print("output directory:", OUT)
print("sections:")
for name, section in result.get("sections", {}).items():
    print(f"- {name}: PASS")
print("\n完整证据文件：")
for path in sorted(OUT.iterdir()):
    print("-", path.name)
print("\n限制：跨断电重建、熵、BER、温度和唯一性仍为 not_measured。")